## Zero-Shot — HMM transferido a Traffic y Exchange

Usa caches HMM entrenados en ETTh1/ETTh2/Weather/Electricity para predecir en Traffic y Exchange **sin re-entrenar el HMM**. Solo se entrena el Transformer.

**Resumible**: salta experimentos ya completados.

In [1]:
import os, sys, time, subprocess
import numpy as np

while not os.path.exists("run.py"):
    os.chdir("..")
sys.path.insert(0, ".")

PRED_LENS = [96, 336]

# ALL 4 source datasets with their optimal HMM config (from K sweep)
SOURCE_CONFIGS = {
    "ETTh1": ("hmm_soft_residual", 4, "etth1"),
    "ETTh2": ("hmm_soft", 8, "etth2"),
    "Weather": ("hmm_soft", 8, "weather"),
    "Electricity": ("hmm_soft_residual", 3, "custom"),
}

# Target zero-shot datasets
TARGETS = {
    "Traffic": {
        "root_path": "./dataset/traffic/",
        "data_path": "traffic.csv",
        "data": "custom",
        "target": "OT",
    },
    "Exchange": {
        "root_path": "./dataset/exchange_rate/",
        "data_path": "exchange_rate.csv",
        "data": "custom",
        "target": "OT",
    },
}

# ALL non-HMM techniques as baselines
NON_HMM_TECHNIQUES = ["patching", "decomposition", "foundation", "text_based", "discretization"]

TRANSFORMER_CFG = {
    "seq_len": 96,
    "label_len": 48,
    "d_model": 64,
    "n_heads": 4,
    "e_layers": 2,
    "d_ff": 128,
    "dropout": 0.1,
    "batch_size": 32,
    "learning_rate": 0.001,
    "lradj": "type1",
    "train_epochs": 10,
    "patience": 3,
}

# Count experiments
n_hmm = len(SOURCE_CONFIGS) * len(TARGETS) * len(PRED_LENS)
n_base = len(NON_HMM_TECHNIQUES) * len(TARGETS) * len(PRED_LENS)
total = n_hmm + n_base
print(f"Total experimentos: {total}")
print(f"  Zero-shot HMM: {n_hmm} (4 sources x 2 targets x 2 horizons)")
print(f"  Baselines: {n_base} (5 techniques x 2 targets x 2 horizons)")

Total experimentos: 36
  Zero-shot HMM: 16 (4 sources x 2 targets x 2 horizons)
  Baselines: 20 (5 techniques x 2 targets x 2 horizons)


In [2]:
def make_des(technique, target_name, source=None, K=None):
    if source and K:
        return f"zeroshot_{target_name}_{source}_{technique}_K{K}"
    return f"zeroshot_{target_name}_{technique}"

def result_path(target_cfg, pred_len, des):
    data = target_cfg["data"]
    cfg = TRANSFORMER_CFG
    seq = cfg["seq_len"]
    ll = cfg["label_len"]
    dm = cfg["d_model"]
    nh = cfg["n_heads"]
    el = cfg["e_layers"]
    df = cfg["d_ff"]
    setting = (
        f"plan_a_{data}_96_{pred_len}_TransformerCommon_{data}"
        f"_ftS_sl{seq}_ll{ll}_pl{pred_len}"
        f"_dm{dm}_nh{nh}_el{el}_dl1"
        f"_df{df}_expand2_dc4_fc1_ebtimeF_dtTrue_{des}_0"
    )
    return f"./results/{setting}/metrics.npy"

def run_exp(target_cfg, pred_len, technique, des, hmm_cache_path=""):
    data = target_cfg["data"]
    cfg = TRANSFORMER_CFG
    cmd = [
        "python", "-u", "run.py",
        "--task_name", "plan_a",
        "--is_training", "1",
        "--root_path", target_cfg["root_path"],
        "--data_path", target_cfg["data_path"],
        "--model_id", f"{data}_96_{pred_len}",
        "--model", "TransformerCommon",
        "--data", data,
        "--features", "S",
        "--target", target_cfg["target"],
        "--seq_len", str(cfg["seq_len"]),
        "--label_len", str(cfg["label_len"]),
        "--pred_len", str(pred_len),
        "--enc_in", "1", "--dec_in", "1", "--c_out", "1",
        "--d_model", str(cfg["d_model"]),
        "--n_heads", str(cfg["n_heads"]),
        "--e_layers", str(cfg["e_layers"]),
        "--d_ff", str(cfg["d_ff"]),
        "--dropout", str(cfg["dropout"]),
        "--batch_size", str(cfg["batch_size"]),
        "--learning_rate", str(cfg["learning_rate"]),
        "--lradj", cfg["lradj"],
        "--train_epochs", str(cfg["train_epochs"]),
        "--patience", str(cfg["patience"]),
        "--use_gpu", "0",
        "--technique", technique,
        "--des", des,
        "--itr", "1",
    ]
    if hmm_cache_path:
        cmd += ["--hmm_cache_path", hmm_cache_path]
        k = int(hmm_cache_path.split("_K")[1].split(".")[0])
        cmd += ["--hmm_k", str(k)]

    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=7200)
    elapsed = time.time() - t0
    mse_line = [l for l in proc.stdout.split("\n") if l.startswith("mse:")]
    if mse_line:
        return elapsed, mse_line[-1]
    err = proc.stderr[-500:] if proc.stderr else proc.stdout[-500:]
    return elapsed, f"ERROR: {err}"


experiments = []

# Baselines per target
for tgt_name, tgt_cfg in TARGETS.items():
    for pl in PRED_LENS:
        for tech in NON_HMM_TECHNIQUES:
            des = make_des(tech, tgt_name)
            experiments.append((tgt_name, tgt_cfg, pl, tech, des, ""))

# Zero-shot HMM per source x target
for src_name, (tech, K, cache_key) in SOURCE_CONFIGS.items():
    cache_path = f"./cache/hmm_{cache_key}_K{K}.pth"
    for tgt_name, tgt_cfg in TARGETS.items():
        for pl in PRED_LENS:
            des = make_des(tech, tgt_name, src_name, K)
            experiments.append((tgt_name, tgt_cfg, pl, tech, des, cache_path))

total = len(experiments)
done = sum(1 for (_, tgt, pl, _, des, _) in experiments if os.path.exists(result_path(tgt, pl, des)))
print(f"Completados: {done}/{total}")

t_start = time.time()
for i, (tgt_name, tgt_cfg, pl, tech, des, cache) in enumerate(experiments, 1):
    rpath = result_path(tgt_cfg, pl, des)
    if os.path.exists(rpath):
        m = np.load(rpath)
        mae, mse = float(m[0]), float(m[1])
        print(f"[{i:>2}/{total}] SKIP {tgt_name} pl={pl} {des} | mse:{mse:.6f}, mae:{mae:.6f}")
        continue
    print(f"[{i:>2}/{total}] RUN  {tgt_name} pl={pl} {des} ...", end=" ", flush=True)
    elapsed, result = run_exp(tgt_cfg, pl, tech, des, cache)
    print(f"{elapsed:.0f}s | {result}")

elapsed_min = (time.time() - t_start) / 60
print(f"\nTotal elapsed: {elapsed_min:.1f}min")

Completados: 12/36
[ 1/36] SKIP Traffic pl=96 zeroshot_Traffic_patching | mse:0.194496, mae:0.280990
[ 2/36] SKIP Traffic pl=96 zeroshot_Traffic_decomposition | mse:0.202261, mae:0.289296
[ 3/36] SKIP Traffic pl=96 zeroshot_Traffic_foundation | mse:0.194038, mae:0.280533
[ 4/36] SKIP Traffic pl=96 zeroshot_Traffic_text_based | mse:0.279811, mae:0.376995
[ 5/36] SKIP Traffic pl=96 zeroshot_Traffic_discretization | mse:0.215431, mae:0.300037
[ 6/36] SKIP Traffic pl=336 zeroshot_Traffic_patching | mse:0.172020, mae:0.260179
[ 7/36] SKIP Traffic pl=336 zeroshot_Traffic_decomposition | mse:0.182663, mae:0.272073
[ 8/36] SKIP Traffic pl=336 zeroshot_Traffic_foundation | mse:0.171930, mae:0.260088
[ 9/36] SKIP Traffic pl=336 zeroshot_Traffic_text_based | mse:0.244108, mae:0.350437
[10/36] SKIP Traffic pl=336 zeroshot_Traffic_discretization | mse:0.187824, mae:0.278990
[11/36] RUN  Exchange pl=96 zeroshot_Exchange_patching ... 72s | mse:0.09947197884321213, mae:0.23634716868400574
[12/36] RUN 

In [3]:
import numpy as np, os, re
from collections import defaultdict

PRED_LENS = [96, 336]
TARGET_NAMES = ["Traffic", "Exchange"]

rows = []
results_dir = "./results"
for d in os.listdir(results_dir):
    # Match pattern: _zeroshot_{TargetName}_{technique}_0
    m = re.search(r"_zeroshot_(Traffic|Exchange)_([\w]+?)_0$", d)
    ds_m = re.match(r"plan_a_(\w+?)_96_(\d+)_", d)
    if not m or not ds_m:
        continue
    mpath = os.path.join(results_dir, d, "metrics.npy")
    if not os.path.exists(mpath):
        continue
    target = m.group(1)
    technique = m.group(2)
    pred_len = int(ds_m.group(2))
    metrics = np.load(mpath)
    mae, mse = float(metrics[0]), float(metrics[1])
    rows.append((target, technique, pred_len, mse, mae))

# Aggregate by target
agg = defaultdict(lambda: defaultdict(list))
for target, technique, pred_len, mse, mae in rows:
    agg[target][technique].append((pred_len, mse, mae))

for tgt in TARGET_NAMES:
    if tgt not in agg:
        print(f"Sin resultados para {tgt}")
        continue
    print(f"\n=== Zero-Shot Results: {tgt} ===")
    header = f"{'Technique':<45}"
    for pl in PRED_LENS:
        header += f"  {'MSE@'+str(pl):>10} {'MAE@'+str(pl):>10}"
    header += f"  {'AVG MSE':>10}"
    print(header)
    print("-" * len(header))
    techs = sorted(agg[tgt].items(), key=lambda x: np.nanmean([r[1] for r in x[1]]))
    for technique, results in techs:
        by_pl = {pl: (mse, mae) for pl, mse, mae in results}
        mse_vals = []
        row = f"{technique:<45}"
        for pl in PRED_LENS:
            if pl in by_pl:
                mse, mae = by_pl[pl]
                row += f"  {mse:>10.6f} {mae:>10.6f}"
                mse_vals.append(mse)
            else:
                row += f"  {'N/A':>10} {'N/A':>10}"
        avg = np.nanmean(mse_vals) if mse_vals else float("nan")
        row += f"  {avg:>10.6f}"
        print(row)


=== Zero-Shot Results: Traffic ===
Technique                                          MSE@96     MAE@96     MSE@336    MAE@336     AVG MSE
-------------------------------------------------------------------------------------------------------
foundation                                       0.194038   0.280533    0.171930   0.260088    0.182984
patching                                         0.194496   0.280990    0.172020   0.260179    0.183258
decomposition                                    0.202261   0.289296    0.182663   0.272073    0.192462
discretization                                   0.215431   0.300037    0.187824   0.278990    0.201627
ETTh1_hmm_soft_residual_K4                       0.254578   0.340035    0.205550   0.298578    0.230064
Weather_hmm_soft_K8                              0.246000   0.324226    0.218171   0.305876    0.232086
ETTh2_hmm_soft_K8                                0.273616   0.356491    0.230441   0.321677    0.252029
Electricity_hmm_soft_residua